In [29]:
# Ajuste os caminhos abaixo conforme a pasta onde você descompactou no Colab ou Drive
raw_data_path = './dados/raw'
clean_data_path = './dados/clean'
immu_path = f"{raw_data_path}/sensor_data/main_data/immu"
labels_path = f"{raw_data_path}/behavior_labels/individual"

In [30]:
import glob
import os
import numpy as np
import pandas as pd

# 1. Definir vacas selecionadas
vacas_selecionadas = ["01", "02", "03", "04", "05", "06", "07", "08", "09", "10"]

for vaca in vacas_selecionadas:
    # Buscar os arquivos CSV de aceleração da vaca (T01, T02...)
    pasta_acc_vaca = os.path.join(immu_path, f"T{vaca}")
    arquivos_acc = sorted(glob.glob(os.path.join(pasta_acc_vaca, f"T{vaca}_*.csv")))

    # Definir o caminho do rótulo correspondente
    caminho_lab = os.path.join(labels_path, f"C{vaca}_0725.csv")

    if not arquivos_acc or not os.path.exists(caminho_lab):
        print(
            f"⚠️ Arquivos da vaca {vaca} (Sensor T{vaca} / Rótulo C{vaca}) não foram encontrados."
        )
        continue

    print(
        f"🔄 Processando {len(arquivos_acc)} arquivo(s) de acelerômetro e rótulos da vaca {vaca}..."
    )

    # Carregar dados do acelerômetro e rótulos
    df_acc = pd.concat([pd.read_csv(f) for f in arquivos_acc], ignore_index=True)
    df_lab = pd.read_csv(caminho_lab)

    # Transforma Classe 4 (Feeding head down) em Classe 3 (Feeding)
    df_lab["behavior"] = df_lab["behavior"].replace({4: 3})
    df_lab["behavior"] = df_lab["behavior"].replace({5: 3})
    df_lab["behavior"] = df_lab["behavior"].replace({6: 3})
    df_lab["behavior"] = df_lab["behavior"].replace({1: 2})
    
    # Remover nulos e filtrar a Classe 0 (Unknown / sem anotação válida)
    df_lab_clean = df_lab.dropna(subset=["behavior"])
    df_lab_clean = df_lab_clean[df_lab_clean["behavior"] != 0].copy()

    # Magnitude da Aceleração Triaxial
    df_acc["acc_mag"] = np.sqrt(
        df_acc["accel_x_mps2"] ** 2
        + df_acc["accel_y_mps2"] ** 2
        + df_acc["accel_z_mps2"] ** 2
    )

    # Janelamento temporal de 10 segundos em AMBOS os DataFrames
    df_acc["janela"] = (df_acc["timestamp"] // 10).astype(int)
    df_lab_clean["janela"] = (df_lab_clean["timestamp"] // 10).astype(int)

    # Agregação de estatísticas do acelerômetro (Engenharia de Atributos)
    df_feat = df_acc.groupby("janela").agg(
        {
            "timestamp": "first",
            "accel_x_mps2": ["mean", "std", "min", "max"],
            "accel_y_mps2": ["mean", "std", "min", "max"],
            "accel_z_mps2": ["mean", "std", "min", "max"],
            "acc_mag": ["mean", "std", "min", "max"],
        }
    )

    # Renomear colunas geradas
    df_feat.columns = [
        "timestamp",
        "x_mean",
        "x_std",
        "x_min",
        "x_max",
        "y_mean",
        "y_std",
        "y_min",
        "y_max",
        "z_mean",
        "z_std",
        "z_min",
        "z_max",
        "mag_mean",
        "mag_std",
        "mag_min",
        "mag_max",
    ]
    df_feat = df_feat.reset_index()

    # Mapear Rótulos por Janela (calcula a moda)
    df_lab_janela = (
        df_lab_clean.groupby("janela")["behavior"]
        .agg(lambda s: s.mode().iloc[0] if not s.mode().empty else np.nan)
        .reset_index()
    )

    # Cruzamento via merge
    df_feat = pd.merge(df_feat, df_lab_janela, on="janela", how="inner")
    df_feat = df_feat.sort_values("janela").reset_index(drop=True)
    df_feat["vaca_id"] = f"T{vaca}"

    # Salvar o arquivo individual limpo
    caminho_saida_vaca = os.path.join(clean_data_path, f"T{vaca}.csv")
    df_feat.to_csv(caminho_saida_vaca, index=False)

    print(
        f"💾 Vaca T{vaca} salva em {caminho_saida_vaca} ({len(df_feat)} janelas rotuladas)."
    )

print("\n✅ Processamento da camada CLEAN concluído com sucesso!")

🔄 Processando 15 arquivo(s) de acelerômetro e rótulos da vaca 01...
💾 Vaca T01 salva em ./dados/clean/T01.csv (7139 janelas rotuladas).
🔄 Processando 15 arquivo(s) de acelerômetro e rótulos da vaca 02...
💾 Vaca T02 salva em ./dados/clean/T02.csv (7119 janelas rotuladas).
🔄 Processando 15 arquivo(s) de acelerômetro e rótulos da vaca 03...
💾 Vaca T03 salva em ./dados/clean/T03.csv (7133 janelas rotuladas).
🔄 Processando 15 arquivo(s) de acelerômetro e rótulos da vaca 04...
💾 Vaca T04 salva em ./dados/clean/T04.csv (7136 janelas rotuladas).
🔄 Processando 15 arquivo(s) de acelerômetro e rótulos da vaca 05...
💾 Vaca T05 salva em ./dados/clean/T05.csv (7110 janelas rotuladas).
🔄 Processando 15 arquivo(s) de acelerômetro e rótulos da vaca 06...
💾 Vaca T06 salva em ./dados/clean/T06.csv (7110 janelas rotuladas).
🔄 Processando 15 arquivo(s) de acelerômetro e rótulos da vaca 07...
💾 Vaca T07 salva em ./dados/clean/T07.csv (7130 janelas rotuladas).
🔄 Processando 15 arquivo(s) de acelerômetro e ró